# dgx-spark-hijinks: Gemma 4 test drive (GCP/Colab G4 - RTX PRO 6000 Blackwell, sm_120)

**Version: MEERKAT** - version names are animals, alphabetical, bumped on every push
(...Jackal = probe matrix sibling notebook, Kangaroo = this one's first cut). If the top
of your copy says an earlier animal, re-open from GitHub.

**What this is:** serve **Gemma 4** (E4B / 12B / 31B) with the campaign's vLLM stack on a
**G4 runtime** (NVIDIA RTX PRO 6000 Blackwell, CC 12.0 / `sm_120`, 96 GB) and bank, per
model: bf16-vs-**NVFP4 KV** serving rows, chat smokes, KV-capacity lines, and optional
quick PPL. 96 GB fits all three models - this is the first box where the whole Gemma 4
ladder runs without capacity gymnastics.

**No compiling on this GPU.** The vLLM `sm_120a` wheel is built by CI
([`build-sm120a-wheel.yml`](https://github.com/jethac/vllm/actions/workflows/build-sm120a-wheel.yml)
on `jethac/vllm`, branch `spark/hijinks-e2-vllm`) and installed from a GitHub Release -
Colab/P520 never compile vLLM again. FlashInfer stays a source tree with runtime JIT
(known-good with the apt CUDA 13 toolkit).

Campaign links:
[repo](https://github.com/jethac/dgx-spark-hijinks/tree/epoch2) |
[wheel releases](https://github.com/jethac/vllm/releases) |
[Gemma 3 1B FlashInfer numerics bug](https://github.com/jethac/dgx-spark-hijinks/blob/epoch2/docs/BUG_FLASHINFER_GEMMA3_1B_SERVING_NUMERICS.md) |
sibling probe notebook: `colab_sm120_probe_matrix.ipynb`


In [ ]:
# Cell 1 - runtime guard. REQUIRE compute capability 12.x; fail fast otherwise.
NOTEBOOK_VERSION = 'MEERKAT'  # KEEP IN SYNC with the banner in the title cell
import subprocess
q = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total,compute_cap',
                    '--format=csv,noheader'], capture_output=True, text=True)
assert q.returncode == 0 and q.stdout.strip(), (
    'nvidia-smi failed - no GPU attached.\n'
    'Runtime -> Change runtime type -> select the G4 GPU '
    '(NVIDIA RTX PRO 6000 Blackwell), then re-run this cell.')
name, driver, mem, cc = [s.strip() for s in q.stdout.strip().splitlines()[0].split(',')]
print(f'GPU: {name} | driver {driver} | {mem} | compute capability {cc}')
assert int(cc.split('.')[0]) == 12, (
    f'Got {name} (CC {cc}) - this notebook is sm_120a-specific and is MEANINGLESS on '
    'A100/L4/T4 runtimes.\nRuntime -> Change runtime type -> pick the G4 runtime '
    '(NVIDIA RTX PRO 6000 Blackwell, CC 12.0, 96 GB), then restart.')
VRAM_GIB = float(mem.split()[0]) / 1024.0
print(f'VRAM: {VRAM_GIB:.1f} GiB ' + ('(G4-class 96 GB: full Gemma 4 ladder fits)'
      if VRAM_GIB >= 80 else '(consumer-class: model menu trimmed below)'))

ALL_MODELS = ['google/gemma-4-E4B-it', 'google/gemma-4-12B-it', 'google/gemma-4-31B-it']
if VRAM_GIB >= 80:
    MODEL_MENU = list(ALL_MODELS)          # 96 GB G4: all three in bf16
elif VRAM_GIB >= 40:
    MODEL_MENU = ALL_MODELS[:2]            # 48 GB class: E4B + 12B
else:
    MODEL_MENU = ALL_MODELS[:1]            # 16 GB consumer Blackwell: E4B only
print('model menu for this box:', MODEL_MENU)


In [ ]:
# Cell 2 - WHOLESALE environment (the campaign's hard-won recipe; deterministic, no
# iterating). ~10-15 min on a fresh runtime, dominated by the apt CUDA toolkit + wheel.
#
# ============================== EDIT ME ======================================
WHEEL_RELEASE_TAG = 'sm120a-wheels-6adc00f70'   # latest green tag from
#       https://github.com/jethac/vllm/releases  (built by build-sm120a-wheel.yml;
#       tag = sm120a-wheels-<9-char shortsha of spark/hijinks-e2-vllm>)
# =============================================================================
VLLM_REPO = 'jethac/vllm'
FLASHINFER_BRANCH = 'spark/hijinks-022-fa2-d512'   # source-tree, runtime JIT (no wheel
#       on purpose: JIT compiles only the kernels actually used, keyed to this GPU;
#       known-good with the apt cuda-toolkit on Colab since notebook DINGO)
HIJINKS_BRANCH = 'epoch2'
TORCH_PIN = 'torch==2.12.0'                         # P520 sm_120 known-good ABI (cu130)
VENV = '/content/g4venv'
import json as _json, os, subprocess, sys, urllib.request

def sh(cmd, check=True, env=None):
    print('+', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    if r.stdout: print(r.stdout[-2000:])
    if r.stderr: print(r.stderr[-2000:])
    if check and r.returncode != 0:
        raise RuntimeError(f'failed ({r.returncode}): {cmd}')
    return r

# 1) full nvcc 13 via NVIDIA apt repo. Wholesale cuda-toolkit-13-0, NOT a minimal
#    package set - the minimal set cost five round-trips (nvcc, nvrtc, cusparse,
#    cublas, ...). Verified dead end: the pip cu13 'nvcc' wheel is ptxas-only.
def _have_nvcc():
    try:
        return subprocess.run(['/usr/local/cuda-13.0/bin/nvcc', '--version'],
                              capture_output=True).returncode == 0
    except FileNotFoundError:
        return False
if not _have_nvcc():
    ub = '2404' if '24.04' in open('/etc/os-release').read() else '2204'
    sh(f'wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu{ub}/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/ck.deb')
    sh('dpkg -i /tmp/ck.deb && apt-get -qq update')
    sh('apt-get -qq install -y cuda-toolkit-13-0')   # ~3 GB / ~8 min, deterministic
CUDA_HOME = '/usr/local/cuda-13.0'
sh(f'{CUDA_HOME}/bin/nvcc --version | tail -1')

# 2) fresh venv - isolates the campaign stack from Colab's preinstalled torch/vllm.
#    (No kernel restart needed: serving runs as venv subprocesses, not in this kernel.)
# virtualenv, not `python -m venv`: Colab's python breaks venv's ensurepip
# bootstrap (returns 1); virtualenv ships its own pip and dodges it.
if not os.path.isfile(f'{VENV}/bin/pip'):
    sh(f'rm -rf {VENV}', check=False)
    sh(f'{sys.executable} -m pip install -q virtualenv')
    sh(f'{sys.executable} -m virtualenv {VENV}')
VPIP = f'{VENV}/bin/pip'
VPY = f'{VENV}/bin/python'
sh(f'{VPIP} install -q --upgrade pip')

# 3) torch cu130, pinned (must go in BEFORE the vLLM wheel so its deps resolve
#    against this torch, not a PyPI cu126 one).
sh(f'{VPIP} install -q {TORCH_PIN} --index-url https://download.pytorch.org/whl/cu130')

# 4) the CI-built sm_120a vLLM wheel from the GitHub Release (THE point of this lane:
#    this cell replaces the 1-2 h Colab source build).
rel = _json.load(urllib.request.urlopen(
    f'https://api.github.com/repos/{VLLM_REPO}/releases/tags/{WHEEL_RELEASE_TAG}'))
whls = [a['browser_download_url'] for a in rel['assets'] if a['name'].endswith('.whl')]
assert whls, (f'no .whl asset on release {WHEEL_RELEASE_TAG} - check '
              f'https://github.com/{VLLM_REPO}/releases  (CI still running? wrong tag?) '
              'Last resort: the SLOW source-build fallback cell below.')
WHEEL_URL = whls[0]
print('wheel:', WHEEL_URL)
sh(f'{VPIP} install -q "{WHEEL_URL}"')
# campaign rule: no packaged FlashInfer alongside the source tree (stale cubin/JIT
# packages were convicted before - WHEEL_CONTAINER_MATRIX r7 post-mortem).
sh(f'{VPIP} uninstall -q -y flashinfer-python flashinfer-jit-cache flashinfer-cubin',
   check=False)

# 5) FlashInfer source tree (campaign branch) on PYTHONPATH, JIT mode + its deps.
if not os.path.isdir('/content/flashinfer'):
    sh(f'git clone --recursive https://github.com/jethac/flashinfer -b {FLASHINFER_BRANCH} /content/flashinfer')
sh(f'{VPIP} install -q ninja jinja2 packaging apache-tvm-ffi filelock requests einops nvidia-ml-py')

# 6) campaign repo: serving/PPL harnesses + the bundled PPL corpus slices.
if not os.path.isdir('/content/dgx-spark-hijinks'):
    sh(f'git clone --depth 5 https://github.com/jethac/dgx-spark-hijinks -b {HIJINKS_BRANCH} /content/dgx-spark-hijinks')
HIJINKS = '/content/dgx-spark-hijinks'
C1_CORPUS = f'{HIJINKS}/results/claude_anomaly_corpus_sweep_20260611/docs/c1_ppl_corpus.md'
assert os.path.isfile(C1_CORPUS), 'bundled C1 corpus slice missing from clone'

# 7) import gate (this box HAS the GPU, unlike the CI runner - safe to import).
ENV_BASE = dict(os.environ,
                CUDA_HOME=CUDA_HOME,
                PATH=f'{CUDA_HOME}/bin:' + os.environ['PATH'],
                PYTHONPATH='/content/flashinfer')
r = sh(f'{VPY} -c "import torch, vllm, flashinfer; '
       'print(\'torch\', torch.__version__, \'| vllm\', vllm.__version__, '
       '\'| flashinfer\', flashinfer.__version__)"', env=ENV_BASE)
print('ENVIRONMENT READY')


<details><summary><b>Fallback: full source build (SLOW, ~1-2 h GPU-time wasted - only
when no release wheel matches)</b></summary>

The cell below rebuilds the sm_120a wheel from source on this runtime. It exists ONLY
for the gap where `spark/hijinks-e2-vllm` moved and CI hasn't published the new tag yet.
Set `RUN_SOURCE_BUILD = True` explicitly; otherwise it refuses to run. Prefer waiting
for CI or re-running with the previous green tag.
</details>


In [ ]:
# FALLBACK (SLOW) - source-build the sm_120a wheel on this runtime. Normally SKIP.
RUN_SOURCE_BUILD = False   # <- flip deliberately; this burns 1-2 h of paid GPU runtime
if not RUN_SOURCE_BUILD:
    print('skipped (RUN_SOURCE_BUILD=False) - using the CI release wheel. Good.')
else:
    import os, subprocess
    sh('apt-get -qq install -y ccache || true', check=False)
    if not os.path.isdir('/content/vllm-src'):
        sh('git clone --depth 50 https://github.com/jethac/vllm -b spark/hijinks-e2-vllm /content/vllm-src')
    # campaign use-existing-torch pattern (P520 build_vllm_hijinks.sh):
    sh(f'cd /content/vllm-src && {VPY} use_existing_torch.py')
    sh(f'{VPIP} install -q -r /content/vllm-src/requirements/build/cuda.txt')
    env = dict(ENV_BASE, TORCH_CUDA_ARCH_LIST='12.0a', VLLM_MAIN_CUDA_VERSION='13.0',
               MAX_JOBS='4', NVCC_THREADS='1')
    sh(f'cd /content/vllm-src && {VPIP} wheel --no-build-isolation --no-deps -w /content/wheels . -v',
       env=env)
    import glob
    whl = glob.glob('/content/wheels/vllm-*.whl')[0]
    sh(f'{VPIP} install -q --force-reinstall --no-deps {whl}')
    print('source-built wheel installed:', whl)


In [ ]:
# Cell 4 - Hugging Face auth (google/gemma-* are gated: accept the license on the HF
# model page first). Prefers the Colab secret HF_TOKEN; falls back to the login widget.
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    ENV_BASE['HF_TOKEN'] = os.environ['HF_TOKEN']
    print('HF token loaded from Colab secret HF_TOKEN')
except Exception as e:
    print('no Colab HF_TOKEN secret (', e, ') - opening login widget')
    from huggingface_hub import notebook_login
    notebook_login()
    tok = None
    try:
        from huggingface_hub import HfFolder
        tok = HfFolder.get_token()
    except Exception:
        pass
    if tok:
        os.environ['HF_TOKEN'] = tok
        ENV_BASE['HF_TOKEN'] = tok


In [ ]:
# Cell 5 - serving helpers: launch `vllm serve`, wait-ready, chat smoke, KV-capacity
# extraction, quick PPL via the campaign harness. One server at a time.
import json, os, re, signal, subprocess, time, urllib.request

RESULTS_DIR = '/content/results_g4_gemma4'
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = []
PORT = 8000
_CUR = {'proc': None}

def stop_server():
    if _CUR['proc'] is not None and _CUR['proc'].poll() is None:
        _CUR['proc'].kill()
    subprocess.run('pkill -f "vllm serve" ; pkill -f api_server', shell=True)
    _CUR['proc'] = None
    time.sleep(8)

def launch(model, kv_mode, max_len, util, extra_args, extra_env, log_path):
    stop_server()
    env = dict(ENV_BASE, **extra_env)
    args = [f'{VENV}/bin/vllm', 'serve', model,
            '--max-model-len', str(max_len),
            '--gpu-memory-utilization', str(util),
            '--port', str(PORT)] + extra_args
    if kv_mode:
        args += ['--kv-cache-dtype', kv_mode]
    print('+', ' '.join(args))
    if extra_env:
        print('  env:', ' '.join(f'{k}={v}' for k, v in extra_env.items()))
    log = open(log_path, 'w')
    _CUR['proc'] = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, env=env)
    return _CUR['proc']

def wait_ready(proc, log_path, timeout_s=2700):
    # generous: first run includes the model download (31B bf16 ~62 GB) + FlashInfer JIT
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            urllib.request.urlopen(f'http://localhost:{PORT}/v1/models', timeout=2)
            return True
        except Exception:
            pass
        if proc.poll() is not None:
            print('SERVER EXITED, log tail:')
            print(open(log_path).read()[-2500:])
            return False
        time.sleep(5)
    print(f'not ready after {timeout_s}s, log tail:')
    print(open(log_path).read()[-2500:])
    return False

def chat(model, prompt, max_tok=96):
    body = json.dumps({'model': model, 'temperature': 0, 'max_tokens': max_tok,
                       'messages': [{'role': 'user', 'content': prompt}]}).encode()
    t0 = time.time()
    r = json.load(urllib.request.urlopen(urllib.request.Request(
        f'http://localhost:{PORT}/v1/chat/completions', body,
        {'Content-Type': 'application/json'}), timeout=300))
    return r['choices'][0]['message']['content'], time.time() - t0

def kv_capacity(log_text):
    m = re.search(r'GPU KV cache size:\s*([0-9,]+)\s*tokens', log_text)
    return int(m.group(1).replace(',', '')) if m else None

def proof_lines(log_text):
    pat = r'FLASHINFER|FLASH_ATTN|VO split|LINEAR_V_SF|GPU KV cache size|kv_cache_dtype|attention backend'
    return [l for l in log_text.splitlines() if re.search(pat, l)][-10:]

def quick_ppl(model, run_id, ctx=2048):
    out = f'{RESULTS_DIR}/{run_id}_ppl.json'
    r = subprocess.run(
        [VPY, f'{HIJINKS}/scripts/vllm_prompt_ppl_sweep.py',
         '--url', f'http://localhost:{PORT}', '--model', model, '--tokenizer', model,
         '--text-file', C1_CORPUS, '--ctx', str(ctx), '--run-id', run_id,
         '--runtime-ref', f'Colab G4 sm_120 notebook {NOTEBOOK_VERSION}',
         '--output', out],
        capture_output=True, text=True, env=ENV_BASE)
    if r.returncode != 0:
        print('PPL FAILED:', (r.stdout + r.stderr)[-1200:])
        return None
    j = json.load(open(out))
    return j['contexts'][0]['score']['mean_nll_nats']

def run_row(model, row_name, kv_mode, max_len, util, extra_args, extra_env,
            run_ppl=False, ppl_ctx=2048):
    tag = f"{model.split('/')[-1]}_{row_name}"
    log_path = f'{RESULTS_DIR}/{tag}_server.log'
    rec = {'model': model, 'row': row_name, 'notebook_version': NOTEBOOK_VERSION,
           'kv_mode': kv_mode or 'auto(bf16)', 'env_knobs': extra_env, 'ready': False,
           'kv_tokens': None, 'chat': None, 'ppl_mean_nll': None}
    proc = launch(model, kv_mode, max_len, util, extra_args, extra_env, log_path)
    rec['ready'] = wait_ready(proc, log_path)
    txt = open(log_path).read()
    rec['kv_tokens'] = kv_capacity(txt)
    rec['proof_lines'] = proof_lines(txt)
    for l in rec['proof_lines']:
        print('  |', l[-160:])
    if rec['ready']:
        reply, dt = chat(model, 'Reply with exactly: g4-ok. Then name the capital of Japan.')
        rec['chat'] = reply
        print(f'  chat ({dt:.1f}s): {reply[:160]!r}')
        if run_ppl:
            rec['ppl_mean_nll'] = quick_ppl(model, f'{tag}_c1_ctx{ppl_ctx}', ppl_ctx)
            print(f'  C1 ctx-{ppl_ctx} mean_nll_nats: {rec["ppl_mean_nll"]}')
    stop_server()
    RESULTS.append(rec)
    json.dump(rec, open(f'{RESULTS_DIR}/{tag}_{NOTEBOOK_VERSION}.json', 'w'), indent=2)
    print(f'=== {tag}: ready={rec["ready"]} kv_tokens={rec["kv_tokens"]} ===')
    return rec


## Serving rows - one cell per model

Two rows per model, knobs documented inline:

| row | knobs | why |
|---|---|---|
| `bf16` | no kv-cache flag, no backend force | **FlashInfer is the default** for Gemma 4 bf16 on CC 12.x via the flipped `VLLM_FLASHINFER_BF16_GEMMA` selector (branch commits `20196b594` + `36c9bbc83`, Gemma-4-family-scoped) - this row also proves the Triton retirement path |
| `nvfp4` | `--kv-cache-dtype nvfp4` + `VLLM_NVFP4_KV_VOSPLIT=1 VLLM_NVFP4_KV_LINEAR_V_SF=1` | FA2 VO-split for the D=512 global heads + linear (non-swizzled) V scale factors - the campaign's NVFP4-KV recipe (vllm#40677 demo) |

`RUN_QUICK_PPL` adds a C1 ctx-2048 prompt-PPL number per row (runtime-friendly slice of
the campaign's ctx-8191 protocol; same harness, same corpus). Set `MODEL_CHOICE` to run
a single model, or leave `'all'` to walk the menu from Cell 1.


In [ ]:
# Cell 6 - chooser + per-model knob table.
MODEL_CHOICE = 'all'        # 'all' | 'google/gemma-4-E4B-it' | 'google/gemma-4-12B-it' | 'google/gemma-4-31B-it'
RUN_QUICK_PPL = True        # C1 ctx-2048 quick PPL per row (a few min/row)

# Per-model serve knobs. --language-model-only: text-only serving (the multimodal
# towers are out of scope on this lane; matches the campaign's Spark rows).
# util 0.85 on 96 GB discrete: a GPU-OOM here just kills the process - no
# unified-memory deadlock risk (that is Spark-specific, INCIDENT_20260609).
MODEL_KNOBS = {
    'google/gemma-4-E4B-it':  dict(max_len=16384, util=0.85, extra=['--language-model-only']),
    'google/gemma-4-12B-it':  dict(max_len=16384, util=0.85, extra=['--language-model-only']),
    'google/gemma-4-31B-it':  dict(max_len=8192,  util=0.90, extra=['--language-model-only']),
}
ROWS = [
    # (row_name, kv_mode, extra_env)
    ('bf16',  None,    {}),   # default-FlashInfer via the flipped selector - no knobs needed
    ('nvfp4', 'nvfp4', {'VLLM_NVFP4_KV_VOSPLIT': '1', 'VLLM_NVFP4_KV_LINEAR_V_SF': '1'}),
]
RUN_LIST = MODEL_MENU if MODEL_CHOICE == 'all' else [MODEL_CHOICE]
print('will run:', RUN_LIST)

def run_model(model):
    if model not in RUN_LIST:
        print(f'skipped ({model} not in RUN_LIST)')
        return
    k = MODEL_KNOBS[model]
    for row_name, kv_mode, extra_env in ROWS:
        run_row(model, row_name, kv_mode, k['max_len'], k['util'], k['extra'],
                extra_env, run_ppl=RUN_QUICK_PPL)


In [ ]:
# Cell 7a - google/gemma-4-E4B-it (MoE; the campaign's rung-0 model; the vllm#40677
# BEFORE/AFTER demo model). bf16 row = flipped-selector FlashInfer; nvfp4 row =
# VO-split + linear V-SF. Expect ~1.78x KV tokens on the nvfp4 row at decode parity.
run_model('google/gemma-4-E4B-it')


In [ ]:
# Cell 7b - google/gemma-4-12B-it (dense, multimodal family member served text-only;
# D=512 global heads -> exercises the VO-split path hardest per parameter). Same two
# rows. On 16 GB consumer boxes this is auto-skipped by the Cell 1 menu.
run_model('google/gemma-4-12B-it')


In [ ]:
# Cell 7c - google/gemma-4-31B-it (the capacity hero of the 96 GB G4: ~62 GB bf16
# weights + room for KV. max_len 8192 / util 0.90 - documented knob row; the KV-token
# line is the headline number to compare bf16 vs nvfp4). First run downloads ~62 GB.
run_model('google/gemma-4-31B-it')


## BONUS - Gemma 3 1B bisect datapoint (open bug)

[`docs/BUG_FLASHINFER_GEMMA3_1B_SERVING_NUMERICS.md`](https://github.com/jethac/dgx-spark-hijinks/blob/epoch2/docs/BUG_FLASHINFER_GEMMA3_1B_SERVING_NUMERICS.md):
at the Gemma 3 1B geometry (d256 / SWA-512 / 1 KV head) the FlashInfer serving path is
silently wrong at long context on the P520 (sm_120): FI bf16 C1 PPL is **+0.221 nats**
over the FLASH_ATTN/HF truth (2.35778), while short-prompt chat stays coherent.
Platform attribution is PENDING - this cell is the **second sm_120 datapoint**: a
truth-reference pair (FLASH_ATTN bf16 vs FlashInfer bf16), C1-style PPL at ctx 8191 on
the same bundled corpus. If this G4 reproduces the +0.22 delta, the bug is sm_120-real
(not P520-environmental); if it is clean here, suspicion moves to the P520 env.
Note: the bf16-FlashInfer default flip is Gemma-4-scoped, so BOTH backends are forced
explicitly here via `VLLM_ATTENTION_BACKEND`.


In [ ]:
# Cell 8 - bisect pair: Gemma 3 1B, FLASH_ATTN bf16 (truth ref) vs FlashInfer bf16,
# C1 PPL at ctx 8191 (the bug doc's exact protocol; ~10-15 min total).
RUN_BISECT = True
P520_REF = {'truth_c1': 2.35778, 'fi_bf16_c1': 2.5788}   # bug doc, 2026-06-12
if not RUN_BISECT:
    print('skipped (RUN_BISECT=False)')
else:
    bisect = {}
    for row_name, backend in [('flashattn_bf16', 'FLASH_ATTN'),
                              ('flashinfer_bf16', 'FLASHINFER')]:
        rec = run_row('google/gemma-3-1b-it', f'bisect_{row_name}', None,
                      max_len=8192, util=0.60, extra_args=[],
                      extra_env={'VLLM_ATTENTION_BACKEND': backend},
                      run_ppl=True, ppl_ctx=8191)
        bisect[row_name] = rec['ppl_mean_nll']
    print()
    print(f"P520 reference : truth C1 {P520_REF['truth_c1']}  |  FI bf16 {P520_REF['fi_bf16_c1']} (+0.221)")
    print(f"this G4        : FLASH_ATTN {bisect['flashattn_bf16']}  |  FLASHINFER {bisect['flashinfer_bf16']}")
    if None not in bisect.values():
        delta = bisect['flashinfer_bf16'] - bisect['flashattn_bf16']
        print(f'FI-vs-truth delta on THIS box: {delta:+.4f} nats')
        if delta > 0.05:
            print('>> REPRODUCES the P520 deviation: second sm_120 datapoint, bug is platform-real.')
        else:
            print('>> CLEAN here: deviation may be P520-environmental - bank either way.')


In [ ]:
# Cell 9 - results summary table + optional save to Drive.
import json, os, shutil
SAVE_TO_DRIVE = False   # True -> copies the results dir to Drive
hdr = f"{'model':28} {'row':22} {'ready':5} {'KV tokens':>12} {'C1 mean_nll':>12}  chat"
print(f'=== Gemma 4 G4 test drive - notebook {NOTEBOOK_VERSION} ===')
print(hdr)
print('-' * len(hdr))
for r in RESULTS:
    kv = f"{r['kv_tokens']:,}" if r['kv_tokens'] else '-'
    nll = f"{r['ppl_mean_nll']:.5f}" if r.get('ppl_mean_nll') is not None else '-'
    chat_s = (r['chat'] or '')[:40].replace('\n', ' ')
    print(f"{r['model'].split('/')[-1]:28} {r['row']:22} {str(r['ready']):5} {kv:>12} {nll:>12}  {chat_s}")
# bf16-vs-nvfp4 capacity ratios per model (the headline):
by = {}
for r in RESULTS:
    by.setdefault(r['model'], {})[r['row']] = r
for model, rows in by.items():
    if 'bf16' in rows and 'nvfp4' in rows and rows['bf16']['kv_tokens'] and rows['nvfp4']['kv_tokens']:
        print(f"{model}: nvfp4/bf16 KV capacity = "
              f"{rows['nvfp4']['kv_tokens'] / rows['bf16']['kv_tokens']:.3f}x")
summary_path = f'{RESULTS_DIR}/SUMMARY_{NOTEBOOK_VERSION}.json'
json.dump(RESULTS, open(summary_path, 'w'), indent=2)
print('saved ->', summary_path)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dst = f'/content/drive/MyDrive/results_g4_gemma4_{NOTEBOOK_VERSION}'
    shutil.copytree(RESULTS_DIR, dst, dirs_exist_ok=True)
    print('copied ->', dst)


## After the run
Hand `results_g4_gemma4/` (or the Drive copy) back to the campaign: drop it into
`results/colab_g4_<date>/` on branch `epoch2` - Claude will fold it into
`docs/RESULTS_LEDGER.md` and the upstream issue drafts (vllm#40677, the Gemma 3 1B
numerics bug). The bisect pair (Cell 8) goes straight into
`docs/BUG_FLASHINFER_GEMMA3_1B_SERVING_NUMERICS.md` as the second sm_120 datapoint.
